# Flight Delay Prediction
### A Machine Learning Web Application for Predicting Flight Delays
**Course:** Bachelor of Computer Application (BCA), Session 2026-2027  
**Institution:** Maharishi School of Engineering & Technology (MSoET), MUIT, Noida  
**Author:** Abhishek (Roll No: 2402031117003)  
**Project Guide:** Ms. Shabda  

---
## 1. Dataset Loading, Cleaning & Target Creation (Code Snippet 3.1)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

# Data loading
data_path = "../data/flights.csv"
if os.path.exists(data_path):
    df = pd.read_csv(data_path, low_memory=False)
    # Keep only operated flights (drop cancelled and diverted)
    df = df[(df["CANCELLED"] == 0) & (df["DIVERTED"] == 0)]
    df = df.dropna(subset=["ARRIVAL_DELAY"]).drop_duplicates()
else:
    print("flights.csv not found locally. Using representative dataset from train.py generator...")
    import sys
    sys.path.append("..")
    from src.train import generate_sample_dataset
    df = generate_sample_dataset(n_samples=50000)

# Target creation: delayed if arrival delay >= 15 min (BTS Standard)
df["DELAYED"] = (df["ARRIVAL_DELAY"] >= 15).astype(int)
df["SCHEDULED_DEP_HOUR"] = df["SCHEDULED_DEPARTURE"] // 100

print("Cleaned shape:", df.shape)
print("Target distribution:")
print(df["DELAYED"].value_counts(normalize=True))

## 2. Feature Partitioning and 70/30 Stratified Train-Test Split (Code Snippet 3.2)

In [ ]:
from sklearn.model_selection import train_test_split

numeric_cols = ["DISTANCE", "SCHEDULED_TIME", "SCHEDULED_DEP_HOUR"]
time_cols = ["MONTH", "DAY_OF_WEEK"]
nominal_cols = ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
feature_cols = numeric_cols + time_cols + nominal_cols

df[time_cols + nominal_cols] = df[time_cols + nominal_cols].astype(str)

X = df[feature_cols]
y = df["DELAYED"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## 3. Multi-Branch ColumnTransformer Preprocessing Pipeline (Code Snippet 3.3)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline(steps=[
    ("scale", StandardScaler())
])

time_pipeline = Pipeline(steps=[
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

nominal_pipeline = Pipeline(steps=[
    ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100))
])

preprocessor = ColumnTransformer(transformers=[
    ("Numeric", numeric_pipeline, numeric_cols),
    ("Time", time_pipeline, time_cols),
    ("Nominal", nominal_pipeline, nominal_cols)
])
print("Preprocessor assembled successfully.")

## 4. Model Training & Evaluation (Code Snippets 3.4 & 3.5)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample", n_jobs=-1, random_state=42)
}

pipelines = {}
for name, clf in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe

def evaluate(pipe, X_test, y_test):
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }

results = pd.DataFrame({name: evaluate(p, X_test, y_test) for name, p in pipelines.items()}).T
print("Model Comparison Results:")
display(results.round(4))

## 5. Hyperparameter Tuning & Serialization (Code Snippets 3.6 & 3.7)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [10, 15, 20],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=pipelines["Random Forest"],
    param_distributions=param_grid,
    n_iter=5, cv=3, scoring="roc_auc", random_state=42, n_jobs=-1
)
random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_
print("Best Parameters:", random_search.best_params_)

os.makedirs("../models", exist_ok=True)
joblib.dump(best_model, "../models/flight_delay_model.pkl")
print("Saved best pipeline to ../models/flight_delay_model.pkl")